# Update a user

## What it does

Changes the name and the email of a user.

## What happens to the saved users

It creates a practice user, changes its name, and deletes it at the end. The update date is set again. The real users stay the same.

## Feature it uses

`updateUser(id, name, email)`, from `logic/update-user.js`.

## Failures it can show

- A number nobody has → `User not found`.
- No name → `Name is empty`.
- An email without `@` → `Email is invalid`.
- Another user has that email → `Email is already used`.
- The user cannot be changed → `Could not update the user`.

Open it with `pnpm jupyter` from `first-app`.

In [1]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent

code = r"""
try {
  const { default: Table } = await import("cli-table3");
  const { createUser } = await import("./logic/create-user.js");
  const { updateUser } = await import("./logic/update-user.js");
  const { deleteUser } = await import("./logic/delete-user.js");

  function printUsers(users) {
    const table = new Table({
      head: ["id", "name", "email", "createdAt", "updatedAt"],
      style: { head: ["cyan", "bold"] },
    });
    for (const user of users) {
      table.push([user.id, user.name, user.email, user.createdAt, user.updatedAt]);
    }
    console.log(table.toString());
  }

  const practice = createUser("Practice", `practice-${Date.now()}@x.com`);
  try {
    // 1. Change the name: a table with the new name and a new updatedAt
    printUsers([updateUser(practice.id, "Practice B", practice.email)]);

    // 2. An id nobody has: "User not found"
    try {
      updateUser(999999, "Nobody", "nobody@x.com");
    } catch (error) {
      console.log(error.message);
    }

    // 3. An empty name: "Name is empty"
    try {
      updateUser(practice.id, "", practice.email);
    } catch (error) {
      console.log(error.message);
    }
  } finally {
    deleteUser(practice.id);
  }
} catch (error) {
  console.log(error.message);
}
"""

if (app / "logic").is_dir():
    env = os.environ.copy()
    env["FORCE_COLOR"] = "1"
    env.pop("NO_COLOR", None)
    completed = subprocess.run(
        ["node", "--no-warnings", "--input-type=module", "-e", code],
        cwd=app,
        capture_output=True,
        text=True,
        env=env,
    )
    print(completed.stdout, end="")
    if completed.returncode != 0:
        print(completed.stderr, end="")
else:
    print("Cannot find first-app.")

┌────┬────────────┬──────────────────────────────┬─────────────────────┬─────────────────────┐
│ id │ name       │ email                        │ createdAt           │ updatedAt           │
├────┼────────────┼──────────────────────────────┼─────────────────────┼─────────────────────┤
│ 22 │ Practice B │ practice-1793027117777@x.com │ 2026-10-26 16:05:17 │ 2026-10-26 16:05:17 │
└────┴────────────┴──────────────────────────────┴─────────────────────┴─────────────────────┘
User not found
Name is empty
